In [1]:
import pandas as pd
import json
import requests
from pydantic import BaseModel, Field, field_validator


# ===== CONFIGURAÇÃO DO LOADER =====
class DataLoaderConfig(BaseModel):
    url: str = Field(..., description="URL do arquivo JSON")
    chave: str | None = Field(None, description="Chave do JSON a ser carregada")

    @field_validator("url")
    @classmethod
    def validar_url(cls, v: str) -> str:
        if not (v.startswith("http://") or v.startswith("https://")):
            raise ValueError("URL deve começar com http:// ou https://")
        return v

    model_config = {
        "extra": "allow",
        "json_schema_extra": {
            "example": {
                "url": "https://exemplo.com/dados.json",
                "chave": "dados_churn",
            }
        },
    }


# ===== FUNÇÃO DE CARREGAMENTO =====
def load_data(config: DataLoaderConfig) -> pd.DataFrame:
    """
    Carrega um JSON da URL e transforma em DataFrame.
    Se houver uma chave, extrai os dados dessa chave.
    """
    response = requests.get(config.url)
    response.raise_for_status()
    data = response.json()

    # Normaliza caso seja dict com chave
    if isinstance(data, dict) and config.chave:
        data = data[config.chave]

    # Caso seja lista ou dict, transforma em DataFrame
    if isinstance(data, list):
        dados_normalizados = pd.json_normalize(data)
    elif isinstance(data, dict):
        dados_normalizados = pd.json_normalize([data])
    else:
        dados_normalizados = pd.DataFrame([data])

    return dados_normalizados


# ===== EXEMPLO DE USO =====
configs = {
    "churn": DataLoaderConfig(
        url="https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dataset-telecon.json",
        chave="dados_telecon"  # ajuste conforme a chave do JSON
    )
}

# Carregar e exibir
dados_normalizados = load_data(configs["churn"])
print("Antes do tratamento:", dados_normalizados.shape)


# ===== TRATAMENTO EXTRA =====
# Substitui espaços em branco em 'conta.cobranca.Total'
idx = dados_normalizados[dados_normalizados['conta.cobranca.Total'] == ' '].index
dados_normalizados.loc[idx, "conta.cobranca.Total"] = (
    dados_normalizados.loc[idx, "conta.cobranca.mensal"] * 24
)
dados_normalizados.loc[idx, "cliente.tempo_servico"] = 24

# Converte coluna numérica
dados_normalizados['conta.cobranca.Total'] = dados_normalizados['conta.cobranca.Total'].astype(float)

# Remove linhas com churn vazio
dados_sem_vazio = dados_normalizados[dados_normalizados['Churn'] != ''].copy()
dados_sem_vazio.reset_index(drop=True, inplace=True)

print("Depois do tratamento:", dados_sem_vazio.shape)
print(dados_sem_vazio.head())


Antes do tratamento: (7344, 21)
Depois do tratamento: (7118, 21)
   id_cliente Churn cliente.genero  cliente.idoso cliente.parceiro  \
0  0002-ORFBO   nao       feminino              0              sim   
1  0003-MKNFE   nao      masculino              0              nao   
2  0004-TLHLJ   sim      masculino              0              nao   
3  0011-IGKFF   sim      masculino              1              sim   
4  0013-EXCHZ   sim       feminino              1              sim   

  cliente.dependentes  cliente.tempo_servico telefone.servico_telefone  \
0                 sim                    9.0                       sim   
1                 nao                    9.0                       sim   
2                 nao                    4.0                       sim   
3                 nao                   13.0                       sim   
4                 nao                    3.0                       sim   

  telefone.varias_linhas internet.servico_internet  ...  \
0         

#Identificação de dados duplicados:

`duplicated()`

In [2]:
# verifica se há amostras duplicadas em nosso conjunto de dados
dados_sem_vazio.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
7113,True
7114,True
7115,True
7116,True


##Quantidade de dados duplicados

In [3]:
dados_sem_vazio.duplicated().sum()

np.int64(75)

###Quais são os dados duplicados?

Para isso, vamos criar um filtro chamado de filtro_duplicadas que vai ser igual à dados_sem_vazio.duplicated(). Para verificar que é o mesmo valor, podemos escrever a variável filtro_duplicadas em uma nova linha.

In [4]:
filtro_duplicadas = dados_sem_vazio.duplicated()
filtro_duplicadas

,0
0,False
1,False
2,False
3,False
4,False
...,...
7113,True
7114,True
7115,True
7116,True


In [5]:
dados_sem_vazio[filtro_duplicadas]

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
7043,0675-NCDYU,nao,feminino,0,sim,sim,72.0,sim,sim,fibra otica,...,sim,sim,sim,sim,sim,dois anos,sim,cartao de credito (automatico),116.40,8543.25
7044,6754-LZUKA,nao,masculino,0,sim,nao,61.0,sim,sim,DSL,...,sim,sim,nao,sim,sim,dois anos,nao,transferencia bancaria (automatica),80.90,4932.50
7045,2192-CKRLV,nao,feminino,0,sim,nao,72.0,nao,sem servico de telefone,DSL,...,sim,sim,nao,nao,sim,dois anos,sim,cheque eletronico,49.20,3580.95
7046,9170-ARBTB,nao,feminino,0,sim,sim,52.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,um ano,nao,cartao de credito (automatico),19.60,1012.40
7047,0447-BEMNG,sim,feminino,0,sim,nao,48.0,nao,sem servico de telefone,DSL,...,nao,sim,nao,nao,sim,mes a mes,sim,transferencia bancaria (automatica),45.30,2145.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,5792-JALQC,nao,feminino,1,nao,nao,52.0,sim,sim,DSL,...,nao,sim,nao,nao,nao,dois anos,nao,transferencia bancaria (automatica),59.85,3103.25
7114,5172-RKOCB,nao,masculino,0,sim,nao,72.0,sim,sim,fibra otica,...,sim,nao,sim,sim,sim,dois anos,sim,cartao de credito (automatico),108.95,7875.00
7115,1934-MKPXS,nao,masculino,0,sim,sim,33.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,um ano,nao,cartao de credito (automatico),20.10,620.55
7116,5959-BELXA,sim,masculino,1,nao,nao,32.0,sim,sim,fibra otica,...,nao,nao,nao,sim,sim,mes a mes,sim,cartao de credito (automatico),96.15,3019.25


##Tratamento de dados duplicados

Podemos citar três motivos principais para a remoção das amostras duplicadas:

*   Viés do modelo: se há amostras duplicadas no conjunto de dados, pode ser que o modelo de machine learning dê mais importância para essas amostras repetidas.
*   Melhora do desempenho do modelo: se inserimos amostras duplicadas, vão ser necessários mais cálculos e poder de processamento, além de ser um desperdício computacional trabalhar com amostras com o mesmo valor e que transmitem a mesma informação.
*   Aumento da qualidade dos resultados: vamos inserir informações únicas, sem dados repetidos. Ou seja, vão ser mais relevantes para o modelo.

A biblioteca Pandas oferece o método drop_duplicates() para retirar amostras duplicadas.

In [6]:
dados_sem_vazio.drop_duplicates(inplace=True)

In [7]:
dados_sem_vazio.duplicated().sum()

np.int64(0)

#Identificando e substituindo dados nulos


`.isna()`

s N/A (sigla para not available) significa "é nulo" ou "não está disponível".

In [8]:
dados_sem_vazio.isna()

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,True,True,True
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7039,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7040,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7041,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


###Quantos e quais são dados nulos?

In [9]:
dados_sem_vazio.isna().sum()

,0
id_cliente,0
Churn,0
cliente.genero,0
cliente.idoso,0
cliente.parceiro,0
cliente.dependentes,0
cliente.tempo_servico,8
telefone.servico_telefone,0
telefone.varias_linhas,0
internet.servico_internet,0


In [10]:
#somar a quantidade total de valores nulos no banco de dados
dados_sem_vazio.isna().sum().sum()

np.int64(114)

In [11]:
dados_sem_vazio[dados_sem_vazio.isna().any(axis=1)]

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0002-ORFBO,nao,feminino,0,sim,sim,9.0,sim,nao,DSL,...,sim,nao,sim,sim,nao,None,None,None,NaN,NaN
9,0016-QLJIS,nao,feminino,0,sim,sim,NaN,sim,sim,DSL,...,sim,sim,sim,sim,sim,dois anos,sim,cheque pelo correio,90.45,5957.90
176,0282-NVSJS,nao,feminino,1,sim,sim,NaN,nao,sem servico de telefone,DSL,...,nao,nao,sim,nao,nao,mes a mes,sim,cheque pelo correio,29.30,355.90
181,0295-QVKPB,nao,masculino,0,nao,nao,NaN,sim,nao,DSL,...,nao,sim,sim,sim,nao,mes a mes,sim,cartao de credito (automatico),63.95,318.10
437,0639-TSIQW,sim,feminino,0,nao,nao,67.0,sim,sim,fibra otica,...,sim,sim,nao,sim,nao,None,None,cartao de credito (automatico),NaN,6886.25
751,1095-WGNGG,nao,feminino,0,sim,nao,NaN,sim,sim,fibra otica,...,sim,nao,nao,sim,sim,dois anos,sim,transferencia bancaria (automatica),101.05,5971.25
963,1396-QWFBJ,sim,feminino,0,sim,sim,21.0,sim,nao,fibra otica,...,sim,nao,nao,nao,nao,None,sim,None,NaN,1565.70
1604,2333-KWEWW,nao,masculino,0,nao,nao,18.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,None,nao,None,20.05,NaN
1605,2335-GSODA,nao,masculino,0,nao,sim,23.0,nao,sem servico de telefone,DSL,...,nao,sim,sim,nao,nao,None,nao,None,NaN,NaN
1606,2338-BQEZT,nao,feminino,0,nao,nao,55.0,sim,nao,DSL,...,nao,nao,nao,nao,nao,None,sim,cartao de credito (automatico),NaN,NaN


##Substituição dos dados nulos

In [12]:
dados_sem_vazio['cliente.tempo_servico'].isna()

,cliente.tempo_servico
0,False
1,False
2,False
3,False
4,False
...,...
7038,False
7039,False
7040,False
7041,False


In [13]:
filtro = dados_sem_vazio['cliente.tempo_servico'].isna()

In [14]:
dados_sem_vazio[filtro][['cliente.tempo_servico', 'conta.cobranca.mensal', 'conta.cobranca.Total']]

,cliente.tempo_servico,conta.cobranca.mensal,conta.cobranca.Total
9,NaN,90.45,5957.90
176,NaN,29.30,355.90
181,NaN,63.95,318.10
751,NaN,101.05,5971.25
3523,NaN,76.10,1054.80
5273,NaN,20.60,116.60
5276,NaN,73.85,3581.40
6134,NaN,69.05,1958.45


ara calcular o valor de "cliente.tempo_servico" da amostra de índice 9, poderíamos dividir 5957.90 por 90.45 que resultaria em um valor quebrado de meses.

In [15]:
5957.90/90.45

65.86954118297402

`np.ceil()`

In [16]:
import numpy as np

# A função np.ceil() arredonda um número para o inteiro mais próximo para cima.
np.ceil(5957.90/90.45)

np.float64(66.0)

Dentro do `np.ceil()`, vamos escrever a coluna dados_sem_vazio['conta.cobranca.Total'], barra de divisão e a coluna dados_sem_vazio['conta.cobranca.mensal']. Com isso, o resultado da divisão será colocado onde tem nulos na coluna "cliente.tempo_servico".

Fora do `np.ceil()`, mas dentro de `fillna()`, vamos colocar o parâmetro inplace igual à True para fazer a modificação inloco, ou seja, no dataframe dados_sem_vazio.

In [17]:
dados_sem_vazio['cliente.tempo_servico'] = dados_sem_vazio['cliente.tempo_servico'].fillna(
    np.ceil(
        dados_sem_vazio['conta.cobranca.Total'] / dados_sem_vazio['conta.cobranca.mensal']
    )
)

In [18]:
dados_sem_vazio[filtro][['cliente.tempo_servico', 'conta.cobranca.mensal', 'conta.cobranca.Total']]

,cliente.tempo_servico,conta.cobranca.mensal,conta.cobranca.Total
9,66.0,90.45,5957.90
176,13.0,29.30,355.90
181,5.0,63.95,318.10
751,60.0,101.05,5971.25
3523,14.0,76.10,1054.80
5273,6.0,20.60,116.60
5276,49.0,73.85,3581.40
6134,29.0,69.05,1958.45


In [19]:
dados_sem_vazio.isna().sum()

,0
id_cliente,0
Churn,0
cliente.genero,0
cliente.idoso,0
cliente.parceiro,0
cliente.dependentes,0
cliente.tempo_servico,0
telefone.servico_telefone,0
telefone.varias_linhas,0
internet.servico_internet,0


----------
### Para saber mais: Tratando dados nulos

**Próxima Atividade**

Em análises e modelos de **machine learning**, é essencial garantir que os dados sejam **precisos e completos**. Um dos problemas mais comuns são os **dados nulos** (faltantes/ausentes), que podem prejudicar tanto a análise quanto o desempenho do modelo.

#### 🔎 O que são dados nulos?

Dados nulos ocorrem quando não há informação disponível para determinada observação. As principais causas incluem:

* falhas no registro dos dados,
* perda de informações,
* erros humanos.

Se não tratados, podem gerar **modelos enviesados ou imprecisos** — e, em alguns casos, impedir que o algoritmo rode.

---

#### ⚙️ Estratégias de tratamento

1. **Exclusão de observações**

   * Útil quando a quantidade de dados nulos é pequena.
   * Evita ruídos, mas pode levar à perda de informações importantes.

2. **Imputação de valores**

   * Substitui valores ausentes por estimativas.
   * Métodos comuns: **média, mediana, moda** ou modelos específicos de imputação.

---

#### 🐼 Pandas na prática

Alguns métodos úteis para identificar nulos em um `DataFrame`:

```python
df.isnull()   # True para valores nulos
df.notnull()  # True para valores válidos
df.isna()     # Igual ao isnull()
df.notna()    # Igual ao notnull()
```

---

#### ✅ Em resumo

Tratar dados nulos é um passo crucial da **preparação de dados**, garantindo:

* **maior precisão** do modelo,
* **boa capacidade de generalização**,
* análises mais **confiáveis e consistentes**.

---

##Valor mais frequente

`value_counts()`

In [20]:
# value_counts() é usado para contar a frequência de valores únicos em uma coluna.
dados_sem_vazio['conta.contrato'].value_counts()

,count
conta.contrato,
mes a mes,3861
dois anos,1688
um ano,1463


Assim, poderíamos pensar em inserir o mes a mes para os valores que são nulos nessa coluna. Porém, não vamos fazer isso porque dessa forma afetaríamos os dados pelos seguintes motivos:

Viés nos dados: o modelo de machine learning vai tentar procurar padrões que vão estar incorretos, pois não são os dados reais. Isso pode levar a previsões enganosas, incorretas e que não são satisfatórias.
Distorção de resultados: inserir um valor que não é o correto faz com que o modelo aprenda com dados incorretos e, consequentemente, levam a previsões que podem ser incorretas.
Por isso, não vamos inserir o valor mais frequente como valor para os dados nulos. Para fazer a inserção dessa forma, precisaríamos fazer análises mais rebuscadas que estão fora do escopo do curso, como análise de regressão e construção de outros modelos de aprendizado não-supervisionado.

##Remoção de dados nulos

In [21]:
colunas_dropar = ['conta.contrato', 'conta.faturamente_eletronico', 'conta.metodo_pagamento']

Já aprendemos como fazer esse filtro, basta fazer dados_sem_vazio[] e passar a variável colunas_dropar que contém as colunas que estamos trabalhando. Fora dos colchetes, acrescentamos isna(). Isso retornaria apenas um dataframe com as três colunas com valores True e False para dados nulos.

Mas, como queremos visualizar as amostras que contém pelo menos uma dessas colunas com valor nulo, vamos acrescentar .any() com axis igual à 1. Isso retornaria uma series booleana. Por fim, acrescentamos .sum() no final do código para visualizar somente a quantidade somada.

In [22]:
dados_sem_vazio[colunas_dropar].isna().any(axis=1).sum()

np.int64(37)

`Método dropna()`

In [23]:
#dropna(), significa "retirar valores nulos"
dados_sem_vazio.dropna(subset=colunas_dropar)

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
1,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.90,542.40
2,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.90,280.85
3,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.00,1237.85
4,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.90,267.40
5,0013-MHZWF,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,...,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.40,571.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,9987-LUTYD,nao,feminino,0,nao,nao,13.0,sim,nao,DSL,...,nao,nao,sim,nao,nao,um ano,nao,cheque pelo correio,55.15,742.90
7039,9992-RRAMN,sim,masculino,0,sim,nao,22.0,sim,sim,fibra otica,...,nao,nao,nao,nao,sim,mes a mes,sim,cheque eletronico,85.10,1873.70
7040,9992-UJOEL,nao,masculino,0,nao,nao,2.0,sim,nao,DSL,...,sim,nao,nao,nao,nao,mes a mes,sim,cheque pelo correio,50.30,92.75
7041,9993-LHIEB,nao,masculino,0,sim,sim,67.0,sim,nao,DSL,...,nao,sim,sim,nao,sim,dois anos,nao,cheque pelo correio,67.85,4627.65


In [24]:
#subset significa literalmente subcolunas
df_sem_nulo = dados_sem_vazio.dropna(subset=colunas_dropar).copy()
df_sem_nulo.head()

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
1,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.9,542.40
2,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.9,280.85
3,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.0,1237.85
4,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.9,267.40
5,0013-MHZWF,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,...,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.4,571.45


###Resetar índices

In [25]:
df_sem_nulo.reset_index(drop=True, inplace=True)

In [26]:
df_sem_nulo.isna().sum()

,0
id_cliente,0
Churn,0
cliente.genero,0
cliente.idoso,0
cliente.parceiro,0
cliente.dependentes,0
cliente.tempo_servico,0
telefone.servico_telefone,0
telefone.varias_linhas,0
internet.servico_internet,0


---

### Para saber mais: Inserindo a moda nos dados

Uma das formas mais comuns de tratar **valores nulos** em um `DataFrame` é a **imputação** — ou seja, substituir os valores ausentes por outros que representem bem o conjunto de dados.

#### 📊 O que é a moda?

A **moda** é a medida estatística que indica o **valor mais frequente** em um conjunto de dados.

* Usar a moda na imputação é simples, eficaz e ajuda a manter a consistência do dataset.

---

#### 💡 Exemplo prático

Imagine um conjunto de dados de vendas com informações de **tamanho de produtos**. Alguns valores estão faltando.
Podemos preenchê-los com a **moda** (o tamanho mais comum):

```python
import pandas as pd

# Criando um DataFrame de exemplo
df = pd.DataFrame({
    'Produto': ['Camisa', 'Calça', 'Tênis', 'Meia', 'Boné'],
    'Tamanho': ['P', 'M', 'M', None, None],
    'Preço': [49.99, 79.99, 199.99, 9.99, 39.99]
})

# Preenchendo valores nulos com a moda
df['Tamanho'].fillna(df['Tamanho'].mode()[0], inplace=True)
print(df)
```

**Resultado:**

| Produto | Tamanho | Preço  |
| ------- | ------- | ------ |
| Camisa  | P       | 49.99  |
| Calça   | M       | 79.99  |
| Tênis   | M       | 199.99 |
| Meia    | M       | 9.99   |
| Boné    | M       | 39.99  |

➡️ Os valores nulos foram substituídos pelo valor mais frequente (**M**).

---

#### 📚 Métodos usados

* [`mode()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mode.html) → retorna o(s) valor(es) mais frequente(s).
* [`fillna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) → substitui valores nulos.

---

#### ✅ Em resumo

* A **imputação pela moda** é útil quando os valores faltantes representam **categorias** (ex.: tamanho, cor, região).
* É uma solução simples para tornar o dataset **mais completo e utilizável**.
* Garante que o modelo ou análise não seja prejudicado por dados ausentes.

---
Perfeito 🙌 Aqui vai a versão curtinha em formato de **dica rápida** para o Colab:

---

### 💡 Dica rápida: Preenchendo nulos com a moda

```python
df['coluna'].fillna(df['coluna'].mode()[0], inplace=True)
```

* `mode()` → retorna o valor mais frequente.
* `fillna()` → substitui os valores nulos.

✅ Útil para colunas **categóricas** (ex.: tamanho, cor, região).
✅ Deixa o dataset **mais completo e consistente**.

---



#Exercício: tratando uma base de dados

Para facilitar a análise dos dados de cadastro de cursos de uma plataforma, você recebeu um arquivo chamado cursos_cadastrados.json. Você foi informado que esses dados podem apresentar problemas que podem prejudicar a análise e interpretação correta dos dados.

Para resolver isso, foi solicitado que você identifique e remova as amostras que apresentam os seguintes problemas:

Valores nulos: são valores ausentes em algumas das colunas que devem ser preenchidos para que a análise seja correta.
Duplicatas: registros iguais que podem prejudicar a análise dos dados, já que estão representando a mesma informação.
Strings vazias: valores de texto que não apresentam nenhum conteúdo escrito, o que pode dificultar a análise de dados, principalmente se a coluna tiver muitos valores assim.
Conversão de tipos: algumas colunas do arquivo podem estar no tipo de dados errado, como texto em vez de número, o que pode prejudicar a análise. Nesse caso você deve realizar a conversão para o tipo correto de cada coluna.
Dentro desse contexto, você precisará de uma abordagem sistemática para limpar os dados. Assim, como isso pode ser feito?

Lembrando que o conteúdo de cursos_cadastrados.json é:

In [31]:
[
    {
        "curso": "Introdução à programação",
        "categoria": "Programação",
        "carga_horaria": "20 horas",
        "concluintes": 100,
        "data_inicio": "2022-01-01",
        "data_conclusao": "2022-01-20",
        "descricao": "Curso introdutório à programação com Python",
        "preco": "99.90",
        "instrutor": {
            "nome": "João Silva",
            "email": "joao.silva@emailaleatorio.com",
            "telefone": "(11) 9999-9999"
        }
    },
    {
        "curso": "Excel para iniciantes",
        "categoria": "Produtividade",
        "carga_horaria": null,
        "concluintes": null,
        "data_inicio": null,
        "data_conclusao": null,
        "descricao": null,
        "preco": null,
        "instrutor": {
            "nome": "Maria Oliveira",
            "email": "maria.oliveira@emailaleatorio.com",
            "telefone": "(11) 8888-8888"
        }
    },
    {
        "curso": "Marketing digital para negócios",
        "categoria": "Marketing",
        "carga_horaria": "30 horas",
        "concluintes": 75,
        "data_inicio": "2022-03-01",
        "data_conclusao": "2022-03-31",
        "descricao": "Curso introdutório em marketing",
        "preco": 89.90,
        "instrutor": {
            "nome": "Ana Santos",
            "email": "ana.santos@emailaleatorio.com",
            "telefone": "(11) 7777-7777"
        }
    },
    {
        "curso": "Inteligência artificial",
        "categoria": "Programação",
        "carga_horaria": "40 horas",
        "concluintes": "",
        "data_inicio": "2022-04-01",
        "data_conclusao": "",
        "descricao": "Curso avançado sobre inteligência artificial com Python",
        "preco": 129.90,
        "instrutor": {
            "nome": "",
            "email": "contato@emailaleatorio.com",
            "telefone": ""
        }
    },
    {
        "curso": "Inglês para negócios",
        "categoria": "Idiomas",
        "carga_horaria": "20 horas",
        "concluintes": 30,
        "data_inicio": "",
        "data_conclusao": "",
        "descricao": "Curso de inglês para negócios",
        "preco": 69.90,
        "instrutor": {
            "nome": "John Smith",
            "email": "",
            "telefone": ""
        }
    },
    {
        "curso": "Introdução à programação",
        "categoria": "Programação",
        "carga_horaria": "20 horas",
        "concluintes": 100,
        "data_inicio": "2022-01-01",
        "data_conclusao": "2022-01-20",
        "descricao": "Curso introdutório à programação com Python",
        "preco": "99.90",
        "instrutor": {
            "nome": "João Silva",
            "email": "joao.silva@emailaleatorio.com",
            "telefone": "(11) 9999-9999"
        }
    }
]

IndentationError: unexpected indent (ipython-input-131464248.py, line 16)

In [30]:
# normaliza o json
df = pd.json_normalize(data)

# exibe o DataFrame resultante
print(df)

NameError: name 'data' is not defined

In [ ]:
df.isnull().sum().sum()

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.duplicated()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df[df['instrutor.nome'] == ""]

In [ ]:
df[df['data_conclusao'] == ""]

In [ ]:
# Substitui strings vazias por valores nulos
df.replace('', pd.NA, inplace=True)

In [ ]:
df.dropna(inplace=True)
print(df)

In [ ]:
df.info()

In [ ]:
# Converte a concluintes para o tipo inteiro
df['concluintes'] = df['concluintes'].astype(int)

# Converte a coluna data_inicio e data_conclusao para o tipo datetime
df['data_inicio'] = pd.to_datetime(df['data_inicio'])
df['data_conclusao'] = pd.to_datetime(df['data_conclusao'])

# Convertendo a coluna preço para o tipo float
df['preco'] = df['preco'].astype(float)

In [ ]:
df.info()